# Stock Trading RL Environment: Training Notebook

**Meta PyTorch OpenEnv Hackathon | Theme 3 (World Modeling) + Theme 4 (Self-Improvement)**

A neural world model generates novel market scenarios. An LLM agent learns to trade against it using GRPO.

**Result: Base model 0.300 -> SFT 0.417 -> GRPO 0.537 (79% improvement)**

| Resource | Link |
|----------|------|
| Live Environment | [HF Space](https://huggingface.co/spaces/sarthakbiswas/stock-trader-env) |
| Best Model | [GRPO neural (0.537)](https://huggingface.co/sarthakbiswas/stock-trader-grpo-neural-model) |
| Market Data | [264K rows, 109 stocks](https://huggingface.co/datasets/sarthakbiswas/stock-trader-market-data) |
| Blog Post | [BLOG.md](https://huggingface.co/spaces/sarthakbiswas/stock-trader-env/blob/main/BLOG.md) |
| GitHub | [Repository](https://github.com/sarthakbiswas97/stock-trader-env/tree/v3/world-model) |

**Stack:** OpenEnv + TRL (GRPOTrainer) + Unsloth + PyTorch

## 1. Setup

Install dependencies and download data from HuggingFace Hub.

In [ ]:
%%capture
!pip install -q openenv-core gymnasium torch pandas numpy matplotlib
!pip install -q huggingface_hub datasets

import os
if not os.path.exists("stock-trader-env"):
    !git clone -q https://github.com/sarthakbiswas97/stock-trader-env.git
os.chdir("stock-trader-env")
!git checkout -q v3/world-model

from huggingface_hub import hf_hub_download
from pathlib import Path
from datasets import load_dataset

ohlcv_dir = Path("data/ohlcv")
if not ohlcv_dir.exists() or not any(ohlcv_dir.glob("*.csv")):
    ds = load_dataset("sarthakbiswas/stock-trader-market-data")
    ohlcv_dir.mkdir(parents=True, exist_ok=True)
    macro_dir = Path("data/macro")
    macro_dir.mkdir(parents=True, exist_ok=True)
    for symbol, group in ds["ohlcv"].to_pandas().groupby("symbol"):
        group.drop(columns=["symbol", "data_type"]).to_csv(ohlcv_dir / f"{symbol}_daily.csv", index=False)
    for name, group in ds["macro"].to_pandas().groupby("symbol"):
        group.drop(columns=["symbol", "data_type"]).to_csv(macro_dir / f"{name}_daily.csv", index=False)

Path("checkpoints/world_model").mkdir(parents=True, exist_ok=True)
hf_hub_download(
    repo_id="sarthakbiswas/stock-trader-market-data",
    filename="best_transformer.pt",
    repo_type="dataset",
    local_dir="checkpoints/world_model",
)

print(f"Market data: {len(list(ohlcv_dir.glob('*.csv')))} stocks")
print("World model checkpoint downloaded")
print("Setup complete")

## 2. The Environment: What the Agent Sees

The environment generates daily market observations with technical indicators. The agent reads text, not numbers.

In [ ]:
import sys
sys.path.insert(0, ".")

from training.gym_wrapper import StockTradingGymEnv

env = StockTradingGymEnv(task_id="single_stock", seed=42, obs_mode="text", simulator_mode="replay")
obs, info = env.reset()

print("=" * 60)
print("OBSERVATION (what the LLM agent sees each day)")
print("=" * 60)
print(obs[:800])
print("...")
print()
print(f"Portfolio value: Rs{info['portfolio_value']:,.0f}")
print(f"Cash: Rs{info['cash']:,.0f}")
print(f"Task: {info['task_id']}, Day 1/{info['total_days']}")
env.close()

## 3. Neural World Model

The causal transformer (1.22M params) generates realistic OHLCV prices. Every episode is stochastic. The agent can't memorize.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from server.neural_simulator import NeuralSimulator

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for idx, seed in enumerate([42, 99]):
    sim = NeuralSimulator(task_id="single_stock", seed=seed, temperature=1.0)
    sim.reset()
    symbol = sim.symbols[0]

    gen_df = sim._generated[symbol]
    lookback = 100
    episode_prices = gen_df["close"].iloc[lookback:].values
    seed_prices = gen_df["close"].iloc[lookback-20:lookback].values

    ax = axes[idx]
    ax.plot(range(-20, 0), seed_prices, "b-", alpha=0.5, label="Real (seed)")
    ax.plot(range(0, len(episode_prices)), episode_prices, "r-", linewidth=2, label="Generated")
    ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
    ax.set_title(f"Neural Env Episode (seed={seed})")
    ax.set_xlabel("Day")
    ax.set_ylabel("Price (Rs)")
    ax.legend()

plt.suptitle("Every episode is different. The agent can't memorize.", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\nWorld Model: {sum(p.numel() for p in sim._model.parameters()):,} parameters")
print(f"Volatility ratio: 0.94x real markets")
print(f"Training time: 7 minutes on single GPU")

## 4. Agent Inference: Before vs After Training

Side-by-side comparison of the base model (can't follow format) vs our best GRPO model (reasons about indicators, makes informed decisions).

Requires GPU for live inference. If no GPU, pre-recorded outputs are shown.

In [ ]:
USE_LIVE_INFERENCE = torch.cuda.is_available()

env = StockTradingGymEnv(task_id="single_stock", seed=42, obs_mode="text", simulator_mode="replay")
sample_obs, _ = env.reset()
env.close()

if USE_LIVE_INFERENCE:
    print("GPU detected, running live inference\n")
    import subprocess
    subprocess.run(["pip", "install", "-q", "unsloth"], capture_output=True)

    from unsloth import FastLanguageModel
    from unsloth.chat_templates import get_chat_template

    model, tokenizer = FastLanguageModel.from_pretrained(
        "sarthakbiswas/stock-trader-grpo-neural-model",
        max_seq_length=1024,
        load_in_4bit=True,
    )
    tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
    FastLanguageModel.for_inference(model)

    SYSTEM_PROMPT = (
        "You are an expert stock trader operating in the Indian equity market.\n"
        "You receive daily market observations with technical indicators and must decide on a trading action.\n\n"
        "Rules:\n"
        "- Respond with EXACTLY one action on the last line\n"
        "- Valid actions: HOLD, BUY, SELL, BUY <SYMBOL>, SELL <SYMBOL>, BUY <SYMBOL> <FRACTION>\n"
        "- Before your action, briefly explain your reasoning in 1-2 sentences inside <think> tags\n\n"
        "Example response:\n"
        "<think>RSI is 25 (oversold) and MACD just turned bullish. Volume is spiking. Good entry point.</think>\n"
        "BUY RELIANCE 0.5"
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Here is today's market data:\n\n{sample_obs}\n\nWhat is your trading action?"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7,
                                 do_sample=True, pad_token_id=tokenizer.eos_token_id)
    trained_response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
else:
    print("No GPU, showing pre-recorded outputs\n")
    trained_response = (
        "<think>RSI at 28.3 is in oversold territory and MACD histogram is turning positive, "
        "suggesting a bullish reversal. Bollinger position near lower band confirms the stock is "
        "undervalued relative to recent range. Volume spike at 1.4x average shows institutional "
        "interest. Risk is moderate with sideways regime.</think>\n"
        "BUY RELIANCE 0.5"
    )

base_response = (
    "Looking at the current market data for RELIANCE, I can see several key indicators:\n\n"
    "The RSI is at 28.3, which suggests the stock is in oversold territory. "
    "The MACD shows a bearish crossover but the histogram is narrowing, which could indicate "
    "a potential reversal. The Bollinger Bands show the price is near the lower band...\n\n"
    "(continues for 500+ tokens with no actionable output)"
)

print("=" * 60)
print("BASE MODEL (DeepSeek 7B zero-shot) | Score: 0.300")
print("=" * 60)
print(base_response)
print()
print("=" * 60)
print("GRPO MODEL (trained against neural env) | Score: 0.537")
print("=" * 60)
print(trained_response)
print()
print("The base model generates analysis but no action (defaults to HOLD).")
print("The trained model reasons concisely and outputs a clear action.")

## 5. Training Evidence: Loss and Reward Curves

Real training logs from actual runs. Not mocked data.

Source files: `results/sft_v3_training_log.json`, `results/grpo_neural_training_log.json`

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

with open("results/sft_v3_training_log.json") as f:
    sft_log = json.load(f)
with open("results/grpo_neural_training_log.json") as f:
    grpo_log = json.load(f)

C_SFT, C_GRPO, C_FAIL, C_GRAY = "#3498db", "#2ecc71", "#e74c3c", "#95a5a6"

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor("white")
for ax in axes.flat:
    ax.set_facecolor("#fafafa")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Panel A: Learning Curve (THE HEADLINE)
models = ["Base\nDeepSeek 7B", "SFT v3\n(distilled)", "GRPO Neural\n(BEST)"]
static = [0.300, 0.399, 0.470]
neural = [0.298, 0.417, 0.537]
x = np.arange(len(models))
w = 0.32
b1 = axes[0,0].bar(x-w/2, static, w, label="Static Env", color=C_SFT, edgecolor="white", linewidth=1.5)
b2 = axes[0,0].bar(x+w/2, neural, w, label="Neural Env", color=C_GRPO, edgecolor="white", linewidth=1.5)
axes[0,0].axhline(y=0.300, color=C_FAIL, linestyle="--", linewidth=1.2, alpha=0.5)
axes[0,0].text(2.45, 0.305, "HOLD floor", fontsize=7.5, color=C_FAIL, alpha=0.7)
for bar in b1:
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.008,
        f"{bar.get_height():.3f}", ha="center", fontsize=9, fontweight="bold", color=C_SFT)
for bar in b2:
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.008,
        f"{bar.get_height():.3f}", ha="center", fontsize=9, fontweight="bold", color="#1a8a4a")
axes[0,0].annotate("", xy=(2.16,0.53), xytext=(0.16,0.30),
    arrowprops=dict(arrowstyle="->", color=C_GRAY, lw=2, connectionstyle="arc3,rad=0.15"))
axes[0,0].text(1.1, 0.48, "+79%", fontsize=11, fontweight="bold", color=C_GRAY, ha="center")
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(models, fontsize=10)
axes[0,0].set_ylabel("Score")
axes[0,0].set_ylim(0.2, 0.62)
axes[0,0].set_title("Result: Base 0.300 -> GRPO 0.537", fontweight="bold")
axes[0,0].legend(fontsize=9, loc="upper left")

# Panel B: SFT Loss
steps = [e["step"] for e in sft_log]
losses = [e["loss"] for e in sft_log]
axes[0,1].plot(steps, losses, color=C_SFT, linewidth=2.5)
axes[0,1].fill_between(steps, losses, alpha=0.08, color=C_SFT)
axes[0,1].axvline(x=200, color=C_GRPO, linestyle="--", alpha=0.7)
best_idx = next(i for i, e in enumerate(sft_log) if e["step"] == 200)
axes[0,1].plot(200, losses[best_idx], "o", color=C_GRPO, markersize=10, zorder=5)
axes[0,1].annotate("Best checkpoint\nscore 0.399", xy=(200, losses[best_idx]),
    xytext=(120, losses[best_idx]+0.15), fontsize=8.5, fontweight="bold", color=C_GRPO,
    arrowprops=dict(arrowstyle="->", color=C_GRPO, lw=1.2))
axes[0,1].set_xlabel("Training Step")
axes[0,1].set_ylabel("Loss")
axes[0,1].set_title("SFT Training Loss (lower loss != better trading)", fontweight="bold")

# Panel C: KL Divergence
g_steps = [e["step"] for e in grpo_log]
kl = [e["kl"] for e in grpo_log]
axes[1,0].plot(g_steps, kl, color=C_GRPO, linewidth=2)
axes[1,0].fill_between(g_steps, kl, alpha=0.1, color=C_GRPO)
axes[1,0].axhline(y=3.0, color=C_FAIL, linestyle="--", linewidth=1.5, alpha=0.7)
axes[1,0].text(155, 3.15, "Previous GRPO v3 collapsed here (KL=4.2)", fontsize=8, color=C_FAIL, fontweight="bold", ha="center")
axes[1,0].fill_between([0, 300], [0, 0], [0.5, 0.5], alpha=0.05, color=C_GRPO)
axes[1,0].text(150, 0.05, "Safe zone (beta=0.05)", fontsize=8, color=C_GRPO, alpha=0.7, ha="center")
axes[1,0].set_xlabel("Training Step")
axes[1,0].set_ylabel("KL Divergence")
axes[1,0].set_title("GRPO KL Divergence (stayed under 0.35)", fontweight="bold")
axes[1,0].set_ylim(-0.05, 3.8)

# Panel D: Trading Reward
r_mean = [e["rewards/trading_reward/mean"] for e in grpo_log]
r_std = [e["rewards/trading_reward/std"] for e in grpo_log]
axes[1,1].fill_between(g_steps, [m-s for m,s in zip(r_mean,r_std)],
    [m+s for m,s in zip(r_mean,r_std)], alpha=0.15, color=C_GRPO, label="Reward std")
axes[1,1].plot(g_steps, r_mean, color=C_GRPO, linewidth=2, label="Trading reward")
if len(r_mean) >= 5:
    smoothed = np.convolve(r_mean, np.ones(5)/5, mode="valid")
    axes[1,1].plot(g_steps[4:], smoothed, color="#27ae60", linewidth=2.5, alpha=0.8, label="Trend (5-step avg)")
axes[1,1].axhline(y=0, color=C_GRAY, linewidth=0.8, alpha=0.5)
axes[1,1].set_xlabel("Training Step")
axes[1,1].set_ylabel("Trading Reward")
axes[1,1].set_title("GRPO Trading Reward Signal", fontweight="bold")
axes[1,1].legend(fontsize=8, loc="lower left")

fig.suptitle("Training Evidence: SFT + GRPO Against Neural Environment",
    fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("results/training_curves_final.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print(f"\nImprovement: Base 0.300 -> SFT 0.417 -> GRPO 0.537")
print(f"Total improvement: +79% over base model (neural env)")
print(f"RL improvement: +29% over SFT (neural env)")
print(f"KL stayed under 0.35 (previous GRPO v3 reached 4.2 and collapsed)")

## 6. Reward Design and Safeguards

The environment prevents reward hacking through multiple mechanisms.

In [ ]:
from server.mistake_tracker import MistakeTracker, MistakeType

print("REWARD DESIGN")
print("=" * 60)
print()
print("1. FORMAT GATE (penalty only)")
print("   Valid format: 0.0 reward (expected, not rewarded)")
print("   Invalid format: -1.0 penalty")
print()
print("2. TRADING REWARD (blended)")
print("   30% step-level: 5D composite P&L with asymmetric penalties")
print("   70% episode-level: aligned with eval grading")
print()
print("3. SIGNAL-BASED HOLD")
print("   HOLD is not free. Missed RSI < 25: -0.15 penalty")
print("   Justified patience (neutral indicators): +0.01")
print()
print("4. MISTAKE TRACKER (7 error types)")
for mt in MistakeType:
    print(f"   - {mt.value}")
print()
print("5. ANTI-HOLD COLLAPSE")
print("   Rolling window: if HOLD > 75%, extra penalty on HOLD")
print("   Bonus for BUY/SELL when HOLD is dominating")
print()
print("These safeguards ensure the agent learns to trade, not to hack the reward.")

## 7. The Failure Gallery

4 GRPO failures, each teaching a lesson about RL reward design.

In [ ]:
failures = [
    {
        "name": "GRPO v1: The HOLD Collapse",
        "score": 0.300,
        "problem": "85% of actions were HOLD. Doing nothing = zero risk.",
        "fix": "Signal-based HOLD: ignoring strong RSI signals now costs -0.15.",
    },
    {
        "name": "GRPO v2: The Format Hack",
        "score": 0.326,
        "problem": "84% of reward came from formatting. 0% from trading.",
        "fix": "Format is now a gate (0/-1), not a reward source.",
    },
    {
        "name": "GRPO v3: The KL Catastrophe",
        "score": 0.301,
        "problem": "Best training metrics, worst eval score. KL hit 4.2.",
        "fix": "KL early stopping. Higher beta (0.05). Shorter training.",
    },
    {
        "name": "SFT v3: The Alignment Tax",
        "score": 0.383,
        "problem": "Step 352 (lower loss) scored WORSE than step 200.",
        "fix": "Select checkpoints by task score, not training loss.",
    },
]

for i, f in enumerate(failures, 1):
    print(f"{'=' * 60}")
    print(f"FAILURE {i}: {f['name']} (score: {f['score']})")
    print(f"{'=' * 60}")
    print(f"  Problem: {f['problem']}")
    print(f"  Fix:     {f['fix']}")
    print()

print("Each failure directly informed the config that scored 0.537.")

## 8. Full Scoreboard: 15 Model Iterations

In [ ]:
scoreboard = [
    ("DeepSeek 7B (base)",    "0.300", "0.298", "No training"),
    ("SFT v1",               "0.300", "-",     "91% HOLD, data imbalance"),
    ("SFT v2",               "0.398", "-",     "Template reasoning"),
    ("GRPO v1",              "~0.300","-",     "Collapsed to HOLD"),
    ("GRPO v2",              "0.326", "-",     "84% reward from formatting"),
    ("GRPO v2.1",            "0.395", "-",     "Reward-aligned, bimodal"),
    ("GRPO v3",              "0.301", "-",     "KL catastrophe (4.2)"),
    ("SFT v3 step-352",      "0.383", "-",     "Overtrained"),
    ("SFT v3 step-200",      "0.399", "0.417", "Distilled reasoning"),
    ("RAFT (from base)",     "0.300", "0.300", "Wrong starting point"),
    ("RAFT v2 (from SFT v3)","0.360", "0.399", "640 winners"),
    ("GRPO neural *",        "0.470", "0.537", "BEST: RL against neural env"),
    ("GRPO v3.1",            "0.418", "0.310", "Improved env"),
    ("GRPO v3.2 ckpt-400",   "-",     "0.485", "Improved env, HOLD% 95->85%"),
    ("GRPO v3.3",            "-",     "0.416", "Lower beta, didn't converge"),
]

print(f"{'#':>2}  {'Model':<25} {'Static':>7} {'Neural':>7}  {'Notes'}")
print("-" * 75)
for i, (name, static, neural, notes) in enumerate(scoreboard, 1):
    marker = " <<" if "BEST" in notes else ""
    print(f"{i:>2}  {name:<25} {static:>7} {neural:>7}  {notes}{marker}")

print()
print("15 iterations. 4 crashes. 1 breakthrough.")
print("Best model: https://huggingface.co/sarthakbiswas/stock-trader-grpo-neural-model")

## 9. GRPO Training Script (Condensed)

The actual training code using TRL GRPOTrainer + Unsloth. This is the condensed version of `scripts/train_grpo.py`.

**Requires A100 GPU and ~1 hour to run.** Shown here for reference. The full script is in the repo.

In [ ]:
# This cell shows the training code structure.
# To actually run training, use: PYTHONPATH=. python scripts/train_grpo.py
# on a machine with A100/4090 GPU.

TRAINING_CODE = '''
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
import json

# 1. Load SFT model with QLoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    "sarthakbiswas/stock-trader-sft-v3-model",
    max_seq_length=1024, load_in_4bit=True,
)

# 2. Load prompts collected from neural environment episodes
prompts = [json.loads(l) for l in open("grpo_improved_prompts.jsonl")]
train_dataset = Dataset.from_list(prompts)  # 1000 prompts from 50 episodes

# 3. Reward functions
def format_gate(completions, **kwargs):
    """Penalty-only: 0 if valid <think>...</think> + ACTION, -1 if invalid"""
    return [0.0 if is_valid(c) else -1.0 for c in completions]

def trading_reward(completions, **kwargs):
    """30% step-level P&L + 70% episode-level return"""
    # Uses forward prices from dataset metadata
    # Asymmetric: bad trades penalized 1.5x vs good trades
    # Anti-HOLD collapse: rolling window penalty
    return [compute_reward(c, kwargs) for c in completions]

# 4. GRPO config (designed to avoid KL catastrophe)
config = GRPOConfig(
    num_generations=4,        # candidates per prompt
    max_completion_length=200,
    beta=0.05,                # KL penalty (v2.1 used 0.01, KL hit 3.9)
    learning_rate=5e-7,
    max_steps=300,            # short to prevent drift
    save_steps=50,            # pick best by eval, not by loss
    scale_rewards="batch",    # Dr. GRPO fix
)

# 5. Train
trainer = GRPOTrainer(
    model=model,
    args=config,
    train_dataset=train_dataset,
    reward_funcs=[format_gate, trading_reward],
    processing_class=tokenizer,
)
trainer.train()
'''

print(TRAINING_CODE)
print("\nFull training script: scripts/train_grpo.py")
print("Full eval script: scripts/eval_sft.py")
print("Models on HF Hub: https://huggingface.co/sarthakbiswas")

## 10. Summary

The environment is the teacher. If the teacher is predictable, the student learns shortcuts.

In [ ]:
print("""
ARCHITECTURE
============

Real Market Data (264K rows, 109 NIFTY stocks, 2015-2026)
        |
        v
Causal Transformer World Model (1.22M params, 4 layers, MDN output)
  Volatility: 0.94x reality. Trained in 7 min.
        |
        v
Generated OHLCV --> Feature Engine --> Text Observation
  (RSI, MACD, Bollinger, trend, momentum, regime, volume)
        |
        v
LLM Agent (DeepSeek-R1 7B, QLoRA r=16)
  Reads text, outputs: <think>reasoning</think> ACTION
        |
        v
Environment Grader (verifiable score 0.0-1.0)


TRAINING PIPELINE
=================

1. SFT: 10K reverse-distilled examples      0.300 -> 0.417
2. RAFT: 640 winning episodes                (slight degradation)
3. GRPO: RL against neural env, 300 steps    0.417 -> 0.537

Total: 79% improvement over base model.
15 iterations. 4 crashes. Lessons in every failure.
""")

print("Live environment: https://huggingface.co/spaces/sarthakbiswas/stock-trader-env")
print("Best model:       https://huggingface.co/sarthakbiswas/stock-trader-grpo-neural-model")
print("Market data:      https://huggingface.co/datasets/sarthakbiswas/stock-trader-market-data")
print("Blog post:        https://huggingface.co/spaces/sarthakbiswas/stock-trader-env/blob/main/BLOG.md")
print("GitHub:           https://github.com/sarthakbiswas97/stock-trader-env")
print()
print("Built by Sarthak Biswas for the Meta PyTorch OpenEnv Hackathon, April 2026.")